# Exploratory Data Analysis: Soccer Event Data
## Focus on Pass Networks and Network Analysis

This notebook explores soccer match event data from multiple competitions including:
- European top leagues (England, Spain, Germany, France, Italy)
- International competitions (World Cup, European Championship)

**Main Goal:** Understand pass patterns and player interactions for network analysis

**Scope:** This EDA analyzes **all competitions** to provide a comprehensive overview. Subsequent network modeling (LE3+) focuses on individual competitions to manage computational complexity.

In [ ]:
# Import necessary libraries
import json
import warnings
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import networkx as nx

warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Create output directory for figures
OUTPUT_DIR = Path('../reports/figures/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {OUTPUT_DIR.absolute()}")

## 1. Load and Understand the Data Structure

In [ ]:
def fix_unicode_escapes(obj):
    """Recursively decode \\uXXXX sequences in all strings."""
    if isinstance(obj, str):
        if r"\u" in obj:  # only bother if there's a unicode escape in there
            try:
                return obj.encode("utf-8").decode("unicode_escape")
            except UnicodeDecodeError:
                return obj  # if something weird happens, leave it as-is
        return obj
    elif isinstance(obj, list):
        return [fix_unicode_escapes(x) for x in obj]
    elif isinstance(obj, dict):
        return {k: fix_unicode_escapes(v) for k, v in obj.items()}
    else:
        return obj


In [ ]:
# Define data paths
data_dir = Path('../data')

def load_json_fixed(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return fix_unicode_escapes(data)

players      = load_json_fixed(data_dir / "players.json")
teams        = load_json_fixed(data_dir / "teams.json")
competitions = load_json_fixed(data_dir / "competitions.json")

# Load event type mappings
event_types = pd.read_csv(data_dir / 'eventid2name.csv')

print("Data loaded successfully!")
print(f"\nCompetitions: {len(competitions)}")
print(f"Teams: {len(teams)}")
print(f"Players: {len(players)}")
print(f"Event types: {len(event_types)}")

In [ ]:
# Convert to DataFrames for easier analysis
df_competitions = pd.DataFrame(competitions)
df_teams = pd.DataFrame(teams)
df_players = pd.DataFrame(players)

print("=== Competitions Overview ===")
print(df_competitions[['name', 'format', 'type']])

print("\n=== Sample Teams ===")
print(df_teams[['name', 'officialName', 'city']].head(10))

In [ ]:
# Event types - focusing on Pass events (event_id = 8)
print("=== Event Types ===")
print(event_types)

# print(event_types.to_latex())


In [ ]:
print("\n=== Pass-related Events (Event ID 8) ===")
pass_events = event_types[event_types['event'] == 8]
print(pass_events)

## 2. Load Event Data from All Competitions

**Note:** For the EDA, we analyze data from all competitions to get a comprehensive overview. In subsequent network analyses (LE3, LE4, etc.), we focus on individual competitions due to computational constraints.

In [ ]:
# Load matches from all competitions for comprehensive analysis
matches_dir = data_dir / 'matches'
all_matches = []

for match_file in sorted(matches_dir.glob('matches_*.json')):
    comp_name = match_file.stem.replace('matches_', '')
    print(f"Loading matches from {comp_name}...")
    with open(match_file, 'r') as f:
        matches = json.load(f)
    
    # Add competition name to each match
    for match in matches:
        match['competition'] = comp_name
    
    all_matches.extend(matches)
    print(f"  Loaded {len(matches)} matches")

df_matches = pd.DataFrame(all_matches)

print(f"\nTotal matches loaded: {len(df_matches)}")
print(f"Match data columns: {df_matches.columns.tolist()}")
print("\nMatches by competition:")
print(df_matches['competition'].value_counts())
print("\nSample match info:")
display(df_matches.sample(3))

In [ ]:
# Count matches per competition
matches_dir = data_dir / 'matches'
match_counts = {}

for match_file in sorted(matches_dir.glob('matches_*.json')):
    comp_name = match_file.stem.replace('matches_', '')
    try:
        with open(match_file, 'r') as f:
            matches = json.load(f)
        match_counts[comp_name] = len(matches)
    except Exception as e:
        print(f"Error loading {comp_name}: {e}")
        match_counts[comp_name] = 0

df_match_counts = pd.Series(match_counts).sort_values(ascending=False)

print("=== Matches per Competition ===")
print(df_match_counts)

# Visualize
plt.figure(figsize=(10, 6))
df_match_counts.plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel('Number of Matches', fontsize=12)
plt.ylabel('Competition', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'matches_per_competition.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Load and parse events data from all competitions
events_dir = data_dir / 'events'
all_events = []

for event_file in sorted(events_dir.glob('events_*.json')):
    comp_name = event_file.stem.replace('events_', '')
    print(f"Loading events from {comp_name}...")
    events = load_json_fixed(event_file)
    
    # Add competition name to each event
    for event in events:
        event['competition'] = comp_name
    
    all_events.extend(events)
    print(f"  Loaded {len(events)} events")

print(f"\nTotal events loaded: {len(all_events)}")

# Convert to DataFrame
df_events = pd.DataFrame(all_events)

print(f"Event data shape: {df_events.shape}")
print(f"Columns: {df_events.columns.tolist()}")
print("\nEvents by competition:")
print(df_events['competition'].value_counts())

In [ ]:
# Explore event structure
print("=== Sample Event Record ===")
print(df_events.iloc[0])

In [ ]:
print("\n=== Event Data Info ===")
print(df_events.info())

## 3. Event Distribution Analysis

In [ ]:
# Visualize event types distribution
event_counts = df_events['eventName'].value_counts().head(15)

plt.figure(figsize=(12, 6))
event_counts.plot(kind='barh', color='skyblue', edgecolor='black')
plt.xlabel('Count', fontsize=12)
plt.ylabel('Event Type', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'event_types_top15.png', dpi=160, bbox_inches='tight')
plt.show()

# Visualize match periods
period_counts = df_events['matchPeriod'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
period_counts.plot(kind='bar', color='lightcoral', edgecolor='black')
plt.xlabel('Match Period', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'events_by_period.png', dpi=160, bbox_inches='tight')
plt.show()

## 4. Deep Dive into Pass Events

In [ ]:
# Calculate minute within period (vectorized)
period_starts = df_events.groupby(['matchId', 'matchPeriod'])['eventSec'].min().reset_index()
period_starts.columns = ['matchId', 'matchPeriod', 'period_start_sec']

df_events = df_events.merge(period_starts, on=['matchId', 'matchPeriod'], how='left')

df_events['minute_in_period'] = ((df_events['eventSec'] - df_events['period_start_sec']) // 60).astype(int)

# Add continuous minutes (vectorized with map)
period_offsets = {'1H': 0, '2H': 45, 'E1': 90, 'E2': 105, 'P': 120}
df_events['minute_continuous'] = df_events['matchPeriod'].map(period_offsets).fillna(0) + df_events['minute_in_period']

# Calculate normalized data (all vectorized operations)
period_minute_counts = df_events.groupby(['matchPeriod', 'minute_in_period']).size().reset_index(name='event_count')
matches_per_period = df_events.groupby('matchPeriod')['matchId'].nunique()

# Add normalized counts and continuous minutes (vectorized)
period_minute_counts['events_per_match'] = period_minute_counts.apply(
    lambda row: row['event_count'] / matches_per_period[row['matchPeriod']], axis=1
)
period_minute_counts['minute_continuous'] = period_minute_counts['matchPeriod'].map(period_offsets).fillna(0) + period_minute_counts['minute_in_period']

# Plot each period separately
plt.figure(figsize=(14, 5))

period_colors = {'1H': 'green', '2H': 'green', 'E1': 'orange', 'E2': 'orange', 'P': 'purple'}
period_order = ['1H', '2H', 'E1', 'E2', 'P']

for period in period_order:
    period_data = period_minute_counts[period_minute_counts['matchPeriod'] == period].sort_values('minute_continuous')
    if len(period_data) > 0:
        plt.plot(period_data['minute_continuous'], period_data['events_per_match'], 
                color=period_colors.get(period, 'green'), linewidth=2, alpha=0.8)

plt.xlabel('Minute (Continuous)', fontsize=12)
plt.ylabel('Events per Match', fontsize=12)
plt.axvline(x=45, color='red', linestyle='--', alpha=0.5, label='Half-time')
plt.axvline(x=90, color='orange', linestyle='--', alpha=0.5, label='Full-time')
plt.axvline(x=105, color='blue', linestyle='--', alpha=0.3, label='End ET1')
plt.axvline(x=120, color='purple', linestyle='--', alpha=0.3, label='End ET2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'events_per_minute_normalized.png', dpi=160, bbox_inches='tight')
plt.show()

# Print sample sizes
print("\nSample sizes by period:")
print(matches_per_period.sort_index())

In [ ]:
# Filter pass events from df_events (eventId == 8)
df_passes = df_events[df_events['eventId'] == 8].copy()

# Add event and subevent labels
df_passes = df_passes.merge(
	event_types[['event', 'subevent', 'event_label', 'subevent_label']],
	left_on=['eventId', 'subEventId'],
	right_on=['event', 'subevent'],
	how='left'
)

# Determine if pass was accurate
df_passes['accurate'] = df_passes['tags'].apply(
	lambda tags: {'id': 1801} in tags if isinstance(tags, list) else False
)

print(f"\nTotal passes: {len(df_passes)}")
print(f"Accurate passes: {df_passes['accurate'].sum()}")
print(f"Pass accuracy: {df_passes['accurate'].mean()*100:.1f}%")

In [ ]:
# Pass accuracy by competition
pass_accuracy = df_passes.groupby('competition')['accurate'].mean() * 100
pass_accuracy = pass_accuracy.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
pass_accuracy.plot(kind='barh', color='teal', edgecolor='black')
plt.xlabel('Accuracy (%)', fontsize=12)
plt.ylabel('Competition', fontsize=12)
plt.xlim(0, 100)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pass_accuracy_by_competition.png', dpi=160, bbox_inches='tight')
plt.show()

# Top passing players - merge with player names
top_passers_ids = df_passes['playerId'].value_counts().head(15)
top_passers_with_names = top_passers_ids.to_frame('pass_count').merge(
	df_players[['wyId', 'shortName']],
	left_index=True,
	right_on='wyId',
	how='left'
).set_index('shortName')['pass_count']

plt.figure(figsize=(12, 7))
top_passers_with_names.plot(kind='barh', color='orange', edgecolor='black')
plt.xlabel('Number of Passes', fontsize=12)
plt.ylabel('Player', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top_passers.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Spatial Analysis of Passes

In [ ]:
# Extract position data
def extract_positions(row):
    """Extract start and end positions from event data"""
    positions = row.get('positions', [])
    if len(positions) >= 2:
        start = positions[0]
        end = positions[1]
        return pd.Series({
            'start_x': start['x'],
            'start_y': start['y'],
            'end_x': end['x'],
            'end_y': end['y']
        })
    return pd.Series({'start_x': None, 'start_y': None, 'end_x': None, 'end_y': None})

# Apply to passes
position_data = df_passes.apply(extract_positions, axis=1)
df_passes = pd.concat([df_passes, position_data], axis=1)

# Remove passes without position data
df_passes_clean = df_passes.dropna(subset=['start_x', 'start_y', 'end_x', 'end_y'])

In [ ]:
# Calculate pass distance and direction
df_passes_clean['pass_distance'] = np.sqrt(
    (df_passes_clean['end_x'] - df_passes_clean['start_x'])**2 + 
    (df_passes_clean['end_y'] - df_passes_clean['start_y'])**2
)

df_passes_clean['pass_angle'] = np.arctan2(
    df_passes_clean['end_y'] - df_passes_clean['start_y'],
    df_passes_clean['end_x'] - df_passes_clean['start_x']
) * 180 / np.pi

# Visualize pass distances
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distance distribution
axes[0, 0].hist(df_passes_clean['pass_distance'], bins=50, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Pass Distance', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Distribution of Pass Distances', fontsize=14, fontweight='bold')
axes[0, 0].axvline(df_passes_clean['pass_distance'].median(), color='red', 
                    linestyle='--', label=f'Median: {df_passes_clean["pass_distance"].median():.1f}')
axes[0, 0].legend()

# Distance by pass type
pass_dist_by_type = df_passes_clean.groupby('subevent_label')['pass_distance'].median().sort_values(ascending=False)
axes[0, 1].barh(pass_dist_by_type.index, pass_dist_by_type.values, color='lightcoral')
axes[0, 1].set_xlabel('Median Distance', fontsize=12)
axes[0, 1].set_ylabel('Pass Type', fontsize=12)
axes[0, 1].set_title('Median Pass Distance by Type', fontsize=14, fontweight='bold')

# Pass origin heatmap
axes[1, 0].hexbin(df_passes_clean['start_x'], df_passes_clean['start_y'], 
                   gridsize=30, cmap='YlOrRd', mincnt=1)
axes[1, 0].set_xlabel('X Position', fontsize=12)
axes[1, 0].set_ylabel('Y Position', fontsize=12)
axes[1, 0].set_title('Pass Origin Heatmap', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim(0, 100)
axes[1, 0].set_ylim(0, 100)

# Pass destination heatmap
axes[1, 1].hexbin(df_passes_clean['end_x'], df_passes_clean['end_y'], 
                   gridsize=30, cmap='YlGnBu', mincnt=1)
axes[1, 1].set_xlabel('X Position', fontsize=12)
axes[1, 1].set_ylabel('Y Position', fontsize=12)
axes[1, 1].set_title('Pass Destination Heatmap', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim(0, 100)
axes[1, 1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pass_distance_analysis.png', dpi=160, bbox_inches='tight')
plt.show()

print("\n=== Pass Distance Statistics ===")
print(df_passes_clean['pass_distance'].describe())

## 6. Player-to-Player Pass Networks

In [ ]:
# PRE-PROCESS: Extract start positions for all events ONCE
def extract_start_position(positions):
    """Extract start x,y from positions array"""
    if positions and len(positions) > 0:
        return positions[0].get('x'), positions[0].get('y')
    return None, None

# Do this ONCE for all events
df_events[['start_x', 'start_y']] = df_events['positions'].apply(
    lambda x: pd.Series(extract_start_position(x))
)

df_passes_accurate = df_passes_clean[df_passes_clean['accurate']].copy()

# Ensure numeric types upfront
df_events['eventSec'] = pd.to_numeric(df_events['eventSec'], errors='coerce')
df_passes_accurate['eventSec'] = pd.to_numeric(df_passes_accurate['eventSec'], errors='coerce')

# VECTORIZED APPROACH: Use merge_asof for temporal joins
def find_receivers_vectorized(passes_df, events_df, distance_threshold=5.0, time_threshold=5.0):
    """Vectorized-ish receiver detection that preserves row alignment"""

    # Prepare events: filter and sort
    events_sorted = events_df[['matchId', 'eventSec', 'playerId', 'teamId', 'start_x', 'start_y']].copy()
    events_sorted = events_sorted.dropna(subset=['start_x', 'start_y'])
    events_sorted = events_sorted.sort_values(['matchId', 'eventSec'])

    # Prepare passes (keep original index!)
    passes_sorted = passes_df[['matchId', 'eventSec', 'playerId', 'teamId', 'end_x', 'end_y']].copy()
    passes_sorted = passes_sorted.sort_values(['matchId', 'eventSec'])

    # This will store receiver per *original* pass index
    receiver_ids = pd.Series(index=passes_sorted.index, dtype='object')

    for match_id in tqdm(passes_sorted['matchId'].unique(), desc="Processing matches"):
        match_passes = passes_sorted[passes_sorted['matchId'] == match_id]
        match_events = events_sorted[events_sorted['matchId'] == match_id]

        for idx, pass_row in match_passes.iterrows():
            # Candidate events after the pass, same team, different player
            candidates = match_events[
                (match_events['eventSec'] > pass_row['eventSec']) &
                (match_events['eventSec'] <= pass_row['eventSec'] + time_threshold) &
                (match_events['teamId'] == pass_row['teamId']) &
                (match_events['playerId'] != pass_row['playerId'])
            ]

            if len(candidates) == 0:
                receiver_ids.loc[idx] = None
                continue

            # Distance from pass end to event start
            distances = np.sqrt(
                (candidates['start_x'] - pass_row['end_x'])**2 +
                (candidates['start_y'] - pass_row['end_y'])**2
            )

            valid_mask = distances <= distance_threshold
            if valid_mask.any():
                closest_idx = distances[valid_mask].idxmin()
                receiver_ids.loc[idx] = candidates.loc[closest_idx, 'playerId']
            else:
                receiver_ids.loc[idx] = None

    # Reindex to original passes_df order just to be extra safe
    return receiver_ids.reindex(passes_df.index)

df_passes_accurate['passer_id'] = df_passes_accurate['playerId']

df_passes_accurate['receiver_id'] = find_receivers_vectorized(
    df_passes_accurate,
    df_events
)

In [ ]:
# Remove passes without receiver
df_passes_network = df_passes_accurate.dropna(subset=['receiver_id'])

print(f"Passes with receiver data: {len(df_passes_network)}")
print(f"Percentage of passes with identified receiver: {len(df_passes_network)/len(df_passes_accurate)*100:.1f}%")

# Create pass counts between players
pass_connections = df_passes_network.groupby(['passer_id', 'receiver_id']).size().reset_index(name='pass_count')

pass_connections = pass_connections.sort_values('pass_count', ascending=False)

# Add other edge attributes to pass_connections
# accuracy
pass_connections = pass_connections.merge(
    df_passes_network.groupby(['passer_id', 'receiver_id'])['accurate'].mean().reset_index(name='accuracy'),
    on=['passer_id', 'receiver_id'],
    how='left'
)

# average pass distance
pass_connections = pass_connections.merge(
    df_passes_network.groupby(['passer_id', 'receiver_id'])['pass_distance'].mean().reset_index(name='avg_pass_distance'),
    on=['passer_id', 'receiver_id'],
    how='left'
)

# average pass angle
pass_connections = pass_connections.merge(
    df_passes_network.groupby(['passer_id', 'receiver_id'])['pass_angle'].mean().reset_index(name='avg_pass_angle'),
    on=['passer_id', 'receiver_id'],
    how='left'
)

display(pass_connections.head(20))

In [ ]:
players_involved = pd.DataFrame(
    set(pass_connections['passer_id']).union(set(pass_connections['receiver_id'])), 
    columns=['player_id']
    )

player_nodes = pd.DataFrame(
    players_involved, columns=['player_id']
    ).merge(
        df_players[['passportArea',
                    'currentTeamId',
                    'weight',
                    'birthDate', 
                    'height', 
                    'role', 
                    'wyId',
                    'foot', 
                    'shortName', 
                    'currentNationalTeamId'
                    ]], 
        left_on='player_id', 
        right_on='wyId', 
        how='left'
        ).merge(
            df_teams[['wyId', 'name']].rename(columns={'wyId': 'currentTeamId', 'name': 'current_team_name'}),
            on='currentTeamId',
            how='left'
        )

player_nodes.rename(
    columns={
        'weight': 'weight_kg',
        'birthDate': 'birth_date',
        'passportArea': 'passport_area',
        'height': 'height_cm',
        'role': 'position_role',
        'foot': 'preferred_foot',
        'shortName': 'player_name',
        'currentNationalTeamId': 'national_team_id'
    }, 
    inplace=True
    )

player_nodes = player_nodes[player_nodes['player_id'] != 0].copy()
player_nodes = player_nodes.set_index('player_id')


player_nodes = player_nodes.drop(columns=['wyId', 'currentTeamId'])

player_nodes['passport_area'] = player_nodes['passport_area'].apply(
    lambda d: d.get('name') if isinstance(d, dict) else d
)

player_nodes['position_role'] = player_nodes['position_role'].apply(
    lambda d: d.get('name') if isinstance(d, dict) else d
)

In [ ]:
G = nx.DiGraph()

# Add nodes + attributes
for pid, attrs in player_nodes.iterrows():
    # Gephi uses 'label' as the visible name
    node_attrs = attrs.to_dict()
    node_attrs['label'] = node_attrs.get('player_name', str(pid))
    G.add_node(pid, **node_attrs)

# Add edges using IDs that match node IDs
for _, row in pass_connections.iterrows():
    s = row['passer_id']
    t = row['receiver_id']
    
    # Skip players not in player_nodes (e.g. id 0)
    if s not in player_nodes.index or t not in player_nodes.index:
        continue

    G.add_edge(s, t, 
               pass_count=row['pass_count'],
               accuracy=row['accuracy'],
               avg_pass_distance=row['avg_pass_distance'],
               avg_pass_angle=row['avg_pass_angle']
              )

# 4) Export to Gephi
nx.write_gexf(G, 'pass_network.gexf')

In [ ]:
# Analyze passes by subevent (pass type)
pass_subtype_stats = df_passes_clean.groupby('subevent_label').agg({
    'id': 'count',
    'pass_distance': 'mean'
}).rename(columns={
    'id': 'total_passes',
    'pass_distance': 'avg_distance'
})

# Calculate success rate per subtype
pass_success_by_subtype = df_passes_clean.groupby(['subevent_label', 'pass_type']).size().unstack(fill_value=0)
if 'successful' in pass_success_by_subtype.columns:
    total = pass_success_by_subtype.sum(axis=1)
    pass_subtype_stats['success_rate'] = (pass_success_by_subtype['successful'] / total * 100)
else:
    pass_subtype_stats['success_rate'] = 0

pass_subtype_stats = pass_subtype_stats.sort_values('total_passes', ascending=False).head(12)

print("=== Pass Types Statistics ===")
print(pass_subtype_stats)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pass volume by type
pass_subtype_stats['total_passes'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Total Passes', fontsize=12)
axes[0].set_ylabel('Pass Type', fontsize=12)
axes[0].set_title('Pass Volume by Type (Top 12)', fontsize=14, fontweight='bold')

# Success rate by type
pass_subtype_stats['success_rate'].plot(kind='barh', ax=axes[1], color='lightgreen')
axes[1].set_xlabel('Success Rate (%)', fontsize=12)
axes[1].set_ylabel('Pass Type', fontsize=12)
axes[1].set_title('Pass Success Rate by Type (Top 12)', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pass_types_analysis.png', dpi=160, bbox_inches='tight')
plt.show()

# Competition-level pass statistics
comp_pass_stats = df_passes_clean.groupby('competition').agg({
    'id': 'count',
    'pass_distance': 'mean'
}).rename(columns={
    'id': 'total_passes',
    'pass_distance': 'avg_distance'
})

# Calculate success rate per competition
pass_success_by_comp = df_passes_clean.groupby(['competition', 'pass_type']).size().unstack(fill_value=0)
if 'successful' in pass_success_by_comp.columns:
    total = pass_success_by_comp.sum(axis=1)
    comp_pass_stats['success_rate'] = (pass_success_by_comp['successful'] / total * 100)
else:
    comp_pass_stats['success_rate'] = 0

comp_pass_stats = comp_pass_stats.sort_values('total_passes', ascending=False)

print("\n=== Pass Statistics by Competition ===")
print(comp_pass_stats)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

comp_pass_stats['total_passes'].plot(kind='barh', ax=axes[0], color='purple', alpha=0.7)
axes[0].set_xlabel('Total Passes', fontsize=12)
axes[0].set_ylabel('Competition', fontsize=12)
axes[0].set_title('Pass Volume by Competition', fontsize=14, fontweight='bold')

comp_pass_stats['success_rate'].plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_xlabel('Success Rate (%)', fontsize=12)
axes[1].set_ylabel('Competition', fontsize=12)
axes[1].set_title('Pass Success Rate by Competition', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 100)

comp_pass_stats['avg_distance'].plot(kind='barh', ax=axes[2], color='coral')
axes[2].set_xlabel('Average Distance', fontsize=12)
axes[2].set_ylabel('Competition', fontsize=12)
axes[2].set_title('Average Pass Distance by Competition', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'competition_pass_stats.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Match-Level Analysis

In [ ]:
# Analyze passes by match period and time
df_passes_clean['match_period'] = df_passes_clean['matchPeriod']
df_passes_clean['event_sec'] = df_passes_clean['eventSec']

# Passes by match period
period_stats = df_passes_clean.groupby('match_period').agg({
    'id': 'count',
    'accurate': 'mean',
    'pass_distance': 'mean'
}).rename(columns={
    'id': 'total_passes',
    'accurate': 'accuracy_rate',
    'pass_distance': 'avg_distance'
})

period_stats['accuracy_rate'] = period_stats['accuracy_rate'] * 100

print("=== Passes by Match Period ===")
print(period_stats)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Passes per period
period_stats['total_passes'].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_xlabel('Match Period', fontsize=12)
axes[0].set_ylabel('Total Passes', fontsize=12)
axes[0].set_title('Pass Volume by Match Period', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

# Accuracy per period
period_stats['accuracy_rate'].plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_xlabel('Match Period', fontsize=12)
axes[1].set_ylabel('Accuracy Rate (%)', fontsize=12)
axes[1].set_title('Pass Accuracy by Match Period', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 100)

# Average distance per period
period_stats['avg_distance'].plot(kind='bar', ax=axes[2], color='salmon')
axes[2].set_xlabel('Match Period', fontsize=12)
axes[2].set_ylabel('Average Distance', fontsize=12)
axes[2].set_title('Average Pass Distance by Match Period', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'passes_by_period.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Team-Level Pass Analysis

In [ ]:
# Team statistics
team_stats = df_passes_clean.groupby('teamId').agg({
    'id': 'count',
    'accurate': 'mean',
    'pass_distance': 'mean',
    'matchId': 'nunique'
}).rename(columns={
    'id': 'total_passes',
    'accurate': 'accuracy_rate',
    'pass_distance': 'avg_distance',
    'matchId': 'matches_played'
})

team_stats['accuracy_rate'] = team_stats['accuracy_rate'] * 100
team_stats['passes_per_match'] = team_stats['total_passes'] / team_stats['matches_played']

# Merge with team names
team_stats = team_stats.merge(
    df_teams[['wyId', 'name']], 
    left_index=True, 
    right_on='wyId', 
    how='left'
)

team_stats = team_stats.sort_values('passes_per_match', ascending=False)

print("=== Top 15 Teams by Passes per Match ===")
print(team_stats[['name', 'passes_per_match', 'accuracy_rate', 'avg_distance']].head(15))

# Visualize top teams
top_15_teams = team_stats.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Passes per match
axes[0].barh(range(len(top_15_teams)), top_15_teams['passes_per_match'], color='dodgerblue')
axes[0].set_yticks(range(len(top_15_teams)))
axes[0].set_yticklabels(top_15_teams['name'])
axes[0].set_xlabel('Passes per Match', fontsize=12)
axes[0].set_ylabel('Team', fontsize=12)
axes[0].set_title('Top 15 Teams by Passes per Match', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Team style comparison: accuracy vs distance
axes[1].scatter(team_stats['avg_distance'], team_stats['accuracy_rate'], 
                s=team_stats['passes_per_match']/5, alpha=0.6, color='crimson')
axes[1].set_xlabel('Average Pass Distance', fontsize=12)
axes[1].set_ylabel('Accuracy Rate (%)', fontsize=12)
axes[1].set_title('Team Playing Style: Pass Distance vs Accuracy\n(size = passes per match)', 
                   fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Temporal Patterns in Passing

In [ ]:
# Analyze passes by match period and time
df_passes_clean['match_period'] = df_passes_clean['matchPeriod']
df_passes_clean['event_sec'] = df_passes_clean['eventSec']

# Passes by match period
period_stats = df_passes_clean.groupby('match_period').agg({
    'id': 'count',
    'accurate': 'mean',
    'pass_distance': 'mean'
}).rename(columns={
    'id': 'total_passes',
    'accurate': 'accuracy_rate',
    'pass_distance': 'avg_distance'
})

period_stats['accuracy_rate'] = period_stats['accuracy_rate'] * 100

print("=== Passes by Match Period ===")
print(period_stats)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Passes per period
period_stats['total_passes'].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_xlabel('Match Period', fontsize=12)
axes[0].set_ylabel('Total Passes', fontsize=12)
axes[0].set_title('Pass Volume by Match Period', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

# Accuracy per period
period_stats['accuracy_rate'].plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_xlabel('Match Period', fontsize=12)
axes[1].set_ylabel('Accuracy Rate (%)', fontsize=12)
axes[1].set_title('Pass Accuracy by Match Period', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 100)

# Average distance per period
period_stats['avg_distance'].plot(kind='bar', ax=axes[2], color='salmon')
axes[2].set_xlabel('Match Period', fontsize=12)
axes[2].set_ylabel('Average Distance', fontsize=12)
axes[2].set_title('Average Pass Distance by Match Period', fontsize=14, fontweight='bold')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 10. Network Metrics Preview (Foundation for Network Analysis)

In [ ]:
# Filter for pass events
df_passes = df_events[df_events['eventId'] == 8].copy()  # Pass event ID = 8

# Classify pass type based on tags
def classify_pass(tags):
    if not isinstance(tags, list):
        return 'unsuccessful'
    
    # Check for interception first (ID 1401)
    if any(tag.get('id') == 1401 for tag in tags):
        return 'intercepted'
    # Check for successful pass (ID 1801)
    elif any(tag.get('id') == 1801 for tag in tags):
        return 'successful'
    # Check for unsuccessful pass (ID 1802)
    elif any(tag.get('id') == 1802 for tag in tags):
        return 'unsuccessful'
    else:
        return 'unsuccessful'

df_passes['pass_type'] = df_passes['tags'].apply(classify_pass)

# Count pass types
print(f"Total passes: {len(df_passes)}")
print(f"Successful passes: {(df_passes['pass_type'] == 'successful').sum()}")
print(f"Unsuccessful passes: {(df_passes['pass_type'] == 'unsuccessful').sum()}")
print(f"Intercepted passes: {(df_passes['pass_type'] == 'intercepted').sum()}")
print(f"Pass success rate: {(df_passes['pass_type'] == 'successful').mean()*100:.1f}%")

# Extract pass positions (start and end)
def extract_positions(row):
    positions = row['positions']
    if len(positions) >= 2:
        return pd.Series({
            'start_x': positions[0]['x'],
            'start_y': positions[0]['y'],
            'end_x': positions[1]['x'],
            'end_y': positions[1]['y']
        })
    else:
        return pd.Series({'start_x': None, 'start_y': None, 'end_x': None, 'end_y': None})

df_passes[['start_x', 'start_y', 'end_x', 'end_y']] = df_passes.apply(extract_positions, axis=1)
df_passes_clean = df_passes.dropna(subset=['start_x', 'start_y', 'end_x', 'end_y'])

# Calculate pass distance
df_passes_clean['pass_distance'] = np.sqrt(
    (df_passes_clean['end_x'] - df_passes_clean['start_x'])**2 + 
    (df_passes_clean['end_y'] - df_passes_clean['start_y'])**2
)

print(f"\nPasses with position data: {len(df_passes_clean)}")
print(f"Average pass distance (all passes): {df_passes_clean['pass_distance'].mean():.2f}")
print(f"Average pass distance (successful): {df_passes_clean[df_passes_clean['pass_type'] == 'successful']['pass_distance'].mean():.2f}")
print(f"Average pass distance (unsuccessful): {df_passes_clean[df_passes_clean['pass_type'] == 'unsuccessful']['pass_distance'].mean():.2f}")
print(f"Average pass distance (intercepted): {df_passes_clean[df_passes_clean['pass_type'] == 'intercepted']['pass_distance'].mean():.2f}")

## 11. Data Preparation Helper Functions for Network Analysis

In [ ]:
def find_pass_receiver(pass_row, all_events_df, distance_threshold=5.0, time_threshold=5.0):
    """
    Find the receiver of a pass by looking at the next action near the pass endpoint.
    Uses the "next action near endpoint" approach.
    
    Works for all pass types: successful, unsuccessful, and intercepted.
    For intercepted passes, the receiver might be from the opposing team.
    
    Parameters:
    -----------
    pass_row : Series
        The pass event row
    all_events_df : DataFrame
        All events in the match
    distance_threshold : float
        Maximum distance from endpoint to consider (default 5.0)
    time_threshold : float
        Maximum time after pass to consider (default 5.0 seconds)
        
    Returns:
    --------
    int or None : Player ID of the receiver, or None if not found
    """
    match_id = pass_row['matchId']
    pass_time = pass_row['eventSec']
    end_x, end_y = pass_row['end_x'], pass_row['end_y']
    passer_id = pass_row['playerId']
    team_id = pass_row['teamId']
    pass_type = pass_row.get('pass_type', 'successful')
    
    # For intercepted passes, look for events from EITHER team
    # For successful/unsuccessful passes, look for same team events
    if pass_type == 'intercepted':
        # Get subsequent events in the same match from any team
        subsequent = all_events_df[
            (all_events_df['matchId'] == match_id) & 
            (all_events_df['eventSec'] > pass_time) &
            (all_events_df['eventSec'] <= pass_time + time_threshold) &
            (all_events_df['playerId'] != passer_id)  # Different player
        ].copy()
    else:
        # Get subsequent events in the same match from same team
        subsequent = all_events_df[
            (all_events_df['matchId'] == match_id) & 
            (all_events_df['eventSec'] > pass_time) &
            (all_events_df['eventSec'] <= pass_time + time_threshold) &
            (all_events_df['teamId'] == team_id) &  # Same team
            (all_events_df['playerId'] != passer_id)  # Different player
        ].copy()
    
    if len(subsequent) == 0:
        return None
    
    # Calculate distance from pass endpoint to event start position
    def get_event_start_position(row):
        positions = row.get('positions', [])
        if len(positions) > 0:
            return positions[0].get('x'), positions[0].get('y')
        return None, None
    
    subsequent['start_x'], subsequent['start_y'] = zip(*subsequent.apply(get_event_start_position, axis=1))
    subsequent = subsequent.dropna(subset=['start_x', 'start_y'])
    
    if len(subsequent) == 0:
        return None
    
    # Calculate distances
    subsequent['distance_from_endpoint'] = np.sqrt(
        (subsequent['start_x'] - end_x)**2 + 
        (subsequent['start_y'] - end_y)**2
    )
    
    # Find closest event within threshold
    nearby = subsequent[subsequent['distance_from_endpoint'] <= distance_threshold]
    
    if len(nearby) == 0:
        return None
    
    # Return player of the closest event
    closest = nearby.loc[nearby['distance_from_endpoint'].idxmin()]
    return closest['playerId']


def create_pass_network_for_match(
    match_id, 
    events_df,      # all events
    passes_df,      # df_passes_clean (has 'pass_type')
    distance_threshold=5.0, 
    time_threshold=5.0
):
    """
    Create a pass network for a specific match using the 
    'next action near endpoint' approach.
    
    Now includes ALL pass types: successful, unsuccessful, and intercepted.
    """

    # Filter for passes in this match from passes_df
    match_passes = passes_df[
        (passes_df['matchId'] == match_id) & 
        (passes_df['eventId'] == 8)        # pass event
    ].copy()
    
    # Ensure position data exists
    if not {'start_x', 'start_y', 'end_x', 'end_y'}.issubset(match_passes.columns):
        def extract_positions(row):
            positions = row.get('positions', [])
            if len(positions) >= 2:
                start = positions[0]
                end = positions[1]
                return pd.Series({
                    'start_x': start['x'], 'start_y': start['y'],
                    'end_x': end['x'], 'end_y': end['y']
                })
            return pd.Series({'start_x': None, 'start_y': None, 'end_x': None, 'end_y': None})
        
        position_data = match_passes.apply(extract_positions, axis=1)
        match_passes = pd.concat([match_passes, position_data], axis=1)
        match_passes = match_passes.dropna(subset=['start_x', 'start_y', 'end_x', 'end_y'])
    
    # Pass distance
    if 'pass_distance' not in match_passes.columns:
        match_passes['pass_distance'] = np.sqrt(
            (match_passes['end_x'] - match_passes['start_x'])**2 + 
            (match_passes['end_y'] - match_passes['start_y'])**2
        )
    
    # Find receivers
    match_passes['passer_id'] = match_passes['playerId']
    match_passes['receiver_id'] = match_passes.apply(
        lambda row: find_pass_receiver(row, events_df, distance_threshold, time_threshold), 
        axis=1
    )
    
    # Remove passes without receiver
    match_passes = match_passes.dropna(subset=['receiver_id'])
    
    # Aggregate passes between players by pass_type
    network = match_passes.groupby(['passer_id', 'receiver_id', 'pass_type']).agg({
        'id': 'count',
        'pass_distance': 'mean'
    }).reset_index()
    
    network.columns = ['passer_id', 'receiver_id', 'pass_type', 'pass_count', 'avg_distance']
    
    return network


def create_team_pass_network(
    team_id, 
    match_id, 
    events_df, 
    passes_df,
    distance_threshold=5.0, 
    time_threshold=5.0
):
    """
    Create a pass network for a specific team in a match.
    Now includes all pass types.
    """
    # Match network from all passes
    match_network = create_pass_network_for_match(
        match_id, events_df, passes_df, distance_threshold, time_threshold
    )
    
    # Filter for team's players
    team_passes = events_df[
        (events_df['matchId'] == match_id) & 
        (events_df['teamId'] == team_id)
    ]
    team_player_ids = team_passes['playerId'].unique()
    
    # Keep only edges where both nodes are team players
    team_network = match_network[
        match_network['passer_id'].isin(team_player_ids) &
        match_network['receiver_id'].isin(team_player_ids)
    ]
    
    return team_network

In [ ]:
# Make sure eventSec is float in both dataframes
for df in (df_events, df_passes_clean):
    df['eventSec'] = pd.to_numeric(df['eventSec'], errors='coerce').astype(float)

sample_match = df_passes_network['matchId'].value_counts().index[0]

# generate network edges for the sample match (now includes all pass types)
sample_network = create_pass_network_for_match(
    sample_match,
    df_events,       # all events
    df_passes_clean  # has 'pass_type'
)


sample_network = sample_network.merge(
    df_players[['wyId', 'shortName']].rename(columns={'wyId': 'passer_id', 'shortName': 'passer_name'}),
    on='passer_id', how='left'
).merge(
    df_players[['wyId', 'shortName']].rename(columns={'wyId': 'receiver_id', 'shortName': 'receiver_name'}),
    on='receiver_id', how='left'
)

print(f"Pass network created for match {sample_match} (includes all pass types)")
print(f"Nodes (players): {len(set(sample_network['passer_id']) | set(sample_network['receiver_id']))}")
print(f"Edges (connections): {len(sample_network)}")
print(f"Total passes: {sample_network['pass_count'].sum()}")
print(f"\nPass type breakdown:")
print(sample_network.groupby('pass_type')['pass_count'].sum())



display(sample_network.head(10))

In [ ]:
# Generate LaTeX tables for eda.tex
print("\n=== LaTeX: Event Types ===")
event_types_latex = event_types.to_latex(index=True)
print(event_types_latex)

print("\n=== LaTeX: Sample Pass Network ===")
sample_network_latex = sample_network.head(10).to_latex(index=True)
print(sample_network_latex)

print("\n=== LaTeX: df_matches Sample (5 rows) ===")
df_matches_sample = df_matches[['label', 'date', 'venue', 'duration']].sample(5)
df_matches_sample_latex = df_matches_sample.to_latex(index=False)
print(df_matches_sample_latex)